In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import os
import numpy as np
from scipy.stats import sem
import scipy
from cycler import cycler
%matplotlib inline

In [ ]:
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"] # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
refining_type = "Standard" # "Standard" | "Increment_Training"
refiner = "Fine_Tuned" # "Fine_Tuned" | "Reverse_Probe"
model_name = "CLIP_ViT_Vision" # CLIP_ViT_Vision | DeiT 
domain = "Base_Fine_Tuned" # "Base_Fine_Tuned" | "Fine_Tuned_Layer_Skipping"
transformation = ["Standard", "Base_Fine_Tuned_Classifier", "Base_Linear_Probe"] # "Standard" | "Base_Fine_Tuned_Classifier" | "Base_Linear_Probe"
indices = [i for i in range(12)]
reps = [i for i in range(1,6)]

In [ ]:
for k in range(6, 7): # len(dataset_name)
    results_path = f"./{model_name}/Data/{refining_type}/{refiner}/{dataset_name[k]}/{domain}" # /Entire_Transformation_Matrix_W"

    # 1-5
    results = {}

    for rep in reps:
        results[rep] = []
        path = f"{results_path}/{rep}/Entire_Transformation_Matrix_W"
        try:
            for filename in os.listdir(path):
                if filename in [".DS_Store", f"Base_Fine_Tuned_Classifier_Results_{rep}.json", f"Base_Linear_Probe_Results_{rep}.json", ".ipynb_checkpoints"]: 
                    continue
                file_path = os.path.join(path, filename)
                if os.path.isfile(file_path):
                    results[rep].append(file_path)
        except FileNotFoundError:
            print(f"Error: The Folder '{path}' was not found.")
        except Exception as e:
            print(f"An error occured: {e}")

        data = []
        for filepath in results[rep]:
            try:
                df = pd.read_json(filepath)
                data.append(df)
            except ValueError as ve:
                print(f"Failed to read JSON from file: {filepath} | Error: {ve}")
            except Exception as e:
                print(f"Other error with file {filepath}: {e}")
        data = sorted(data, key=lambda df: df["Train_Data_Size"][0])
        
        results[rep] = data

In [ ]:
mean_results = {}
top_error_bar = {}
bottom_error_bar = {}

for indice in indices:
    mean_results[indice] = []
    top_error_bar[indice] = []
    bottom_error_bar[indice] = []
    for file in range(len(results[1])):
        temp_acc = []
        for rep in range(1, len(results)+1):
            temp_acc.append(results[rep][file]["Classification_Accuracy"][indice])
        ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(temp_acc)-1, loc=np.mean(temp_acc), scale=sem(temp_acc))
        bottom_error_bar[indice].append(ci_low)
        top_error_bar[indice].append(ci_high)
        mean_results[indice].append(np.mean(temp_acc))

In [ ]:
fine_tune_mean = 0
fine_tune_top_error = 0
fine_tune_bottom_error = 0
for k in range(6, 7):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])
    fine_tune_mean = np.mean(refined_acc)

    fine_tune_bottom_error, fine_tune_top_error = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=fine_tune_mean, scale=sem(refined_acc))

In [ ]:
linear_probe_mean = 0
linear_probe_top_error = 0
linear_probe_bottom_error = 0
for k in range(6, 7):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/{refining_type}/Refined_Accuracy/Linear_Probe/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])
    linear_probe_mean = np.mean(refined_acc)

    linear_probe_bottom_error, linear_probe_top_error = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=fine_tune_mean, scale=sem(refined_acc))

In [ ]:
train_size = []
for i in range(len(results[1])):
    train_size.append(results[1][i]["Train_Data_Size"][0][0])

my_colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        "#528c4b", '#e377c2', '#7f7f7f', "#97b47c", '#17becf',
        '#a6cee3', '#b2df8a', '#fb9a99', "#c77b18", "#658a92"
]

plt.rcParams['axes.prop_cycle'] = cycler(color=my_colors)

plt.plot([0, train_size[-1]], [fine_tune_mean, fine_tune_mean], label="Fine Tuned", linestyle="--")
plt.plot([0, train_size[-1]], [linear_probe_mean, linear_probe_mean], label="Linear Probe", linestyle="--")
plt.plot([0, train_size[-1]], [1/397, 1/397], label="Base (Random)", linestyle='--')

for i in indices:
    plt.plot(train_size, mean_results[i], marker="o", label=f"Layer {i}")


plt.xlabel("Number of Training Images")
plt.ylabel("Classification Accuracy (%)")
plt.legend(
    loc="center left",
    bbox_to_anchor=(1.0, 0.5)
)
plt.title(f"Task Matrices: CLIP ViT B/32 Vision - SUN397")
plt.savefig("./CLIP_ViT_Vision_SUN397", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Dataset groups
text_datasets = ['BLiMP', 'HANS', 'TREC-6']
vision_datasets = ['SUN397', 'GTSRB', 'MNIST']

# Data (aligned by dataset)
base =        [15.3, 63.4, 42.5, 65.3, 45.5, 48.9]
linear_probe =[38.1, 76, 75.1, 73.8, 86.8, 98.7]
task_matrix = [50, 82.3, 84.7, 74.8, 87.2, 99.03]
fine_tuned =  [60.5, 99.4, 93.2, 74.5, 98.7, 99.4]
baseline_random = [1/67 * 100, 1/2 * 100, 1/6 * 100, 1/397 * 100, 1/43 * 100, 1/10 * 100]

# Setup
fig, axs = plt.subplots(1, 2, figsize=(9, 7), sharey=True)

bar_width = 0.2

def plot_group(ax, indices, labels, remove_spine):
    x = np.arange(len(indices)) * 0.3

    colors = {
        'Base': 'skyblue',
        'Linear Probe': '#1f78b4',
        'Task Matrix': '#08306b',
        'Fine-Tuned': '#66c2a5'
    }

    for i, idx in enumerate(indices):
        # Values capped at 100
        vals = {
            'Base': min(base[idx], 100),
            'Linear Probe': min(linear_probe[idx], 100),
            'Task Matrix': min(task_matrix[idx], 100),
            'Fine-Tuned': min(fine_tuned[idx], 100)
        }

        # Draw bars in the specific order (back to front)
        order = ['Fine-Tuned', 'Task Matrix', 'Linear Probe', 'Base']

        for label in order:
            ax.bar(x[i], vals[label], width=bar_width, color=colors[label],
                   label=label if (i == 0) else "", alpha=1.0, zorder=order.index(label)+1)

        # Baseline line on top
        left = x[i] - bar_width / 2
        right = x[i] + bar_width / 2
        ax.hlines(y=baseline_random[idx], xmin=left, xmax=right, colors='red', linestyles='dashed',
                  label='Baseline (Random)' if i == 0 else "", linewidth=1.5, zorder=10)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=14)
    ax.set_ylim(0, 105)
    ax.grid(axis='y', linestyle='--', alpha=0.6)

    # Remove specific spines
    if remove_spine == 'right':
        ax.spines['right'].set_visible(False)
    elif remove_spine == 'left':
        ax.spines['left'].set_visible(False)

    for spine in ax.spines.values():
        spine.set_color('lightgrey')

    ax.tick_params(axis='both', which='both', length=0)

# Plot text datasets (left subplot)
plot_group(axs[0], [0, 1, 2], text_datasets, remove_spine='right')
axs[0].set_title("allMiniLM-L12-V2", fontsize=12)

# Plot vision datasets (right subplot)
plot_group(axs[1], [3, 4, 5], vision_datasets, remove_spine='left')
axs[1].set_title("CLIP ViT-B/32 Vision", fontsize=12)

# Shared Y label (grey)
fig.text(0.02, 0.5, 'Accuracy (%)', va='center', rotation='vertical', fontsize=13, color='grey')

# Legend (bottom)
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=5, frameon=True, fontsize=12)

plt.tight_layout(rect=[0.03, 0, 1, 1])
plt.savefig("./addp_results", dpi=600, bbox_inches='tight')
plt.show()